In [39]:
# Cell 1: Environment setup
import os
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"
os.environ["TORCH_USE_CUDA_DSA"] = "1"

In [40]:
# Cell 2: Imports
import os
import torch
import numpy as np
import pandas as pd
import soundfile as sf
import torchaudio
import random
from torch.utils.data import Dataset, DataLoader
from transformers import Wav2Vec2ForSequenceClassification, Wav2Vec2FeatureExtractor

In [41]:
# Cell 3: Check GPU
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")

PyTorch version: 2.12.0.dev20260312+cu128
CUDA available: True
GPU: NVIDIA GeForce RTX 5070 Laptop GPU


In [ ]:
# Cell 4: Load ASVspoof data + WhatsApp voice notes
import os

# Paths
ASVSPOOF_ROOT = os.environ.get("ASVSPOOF_ROOT", r"C:\deepfake-project\data\asvspoof")
PROTOCOL_DIR  = os.path.join(ASVSPOOF_ROOT, "ASVspoof2019_LA_cm_protocols")
TRAIN_AUDIO   = os.path.join(ASVSPOOF_ROOT, "ASVspoof2019_LA_train", "flac")
DEV_AUDIO     = os.path.join(ASVSPOOF_ROOT, "ASVspoof2019_LA_dev", "flac")

# Path to collected WhatsApp voice notes
WHATSAPP_VOICE_DIR = r"C:\whatsapp-voice-bot\ChrisKelleher1947.github.io\bot\collected_voice_notes"

# Verify paths exist
for path_name, path in [("ASVSPOOF_ROOT", ASVSPOOF_ROOT), ("PROTOCOL_DIR", PROTOCOL_DIR), 
                         ("TRAIN_AUDIO", TRAIN_AUDIO), ("DEV_AUDIO", DEV_AUDIO)]:
    if not os.path.exists(path):
        raise FileNotFoundError(f"{path_name} does not exist: {path}")

if not os.path.exists(WHATSAPP_VOICE_DIR):
    print(f"WARNING: WhatsApp voice directory not found: {WHATSAPP_VOICE_DIR}")
    print("Proceeding with ASVspoof data only")
 
# Read train and dev protocol files
train_df = pd.read_csv(
    os.path.join(PROTOCOL_DIR, "ASVspoof2019.LA.cm.train.trn.txt"),
    sep=" ", header=None,
    names=["speaker", "file_id", "env", "attack", "label"]
)
 
dev_df = pd.read_csv(
    os.path.join(PROTOCOL_DIR, "ASVspoof2019.LA.cm.dev.trl.txt"),
    sep=" ", header=None,
    names=["speaker", "file_id", "env", "attack", "label"]
)
 
# Convert labels to integers — 0 = bonafide (REAL), 1 = spoof (FAKE)
train_df["label"] = train_df["label"].map({"bonafide": 0, "spoof": 1})
dev_df["label"]   = dev_df["label"].map({"bonafide": 0, "spoof": 1})
 
# Add the full file path for each audio file
train_df["path"] = train_df["file_id"].apply(lambda x: os.path.join(TRAIN_AUDIO, f"{x}.flac"))
dev_df["path"]   = dev_df["file_id"].apply(lambda x: os.path.join(DEV_AUDIO,   f"{x}.flac"))

# Add WhatsApp voice notes if directory exists
if os.path.exists(WHATSAPP_VOICE_DIR):
    whatsapp_files = [f for f in os.listdir(WHATSAPP_VOICE_DIR) if f.endswith('.ogg')]
    
    whatsapp_df = pd.DataFrame({
        "speaker": ["chris"] * len(whatsapp_files),
        "file_id": [os.path.splitext(f)[0] for f in whatsapp_files],
        "env": ["whatsapp"] * len(whatsapp_files),
        "attack": ["-"] * len(whatsapp_files),
        "label": [0] * len(whatsapp_files),  # All are bonafide (REAL)
        "path": [os.path.join(WHATSAPP_VOICE_DIR, f) for f in whatsapp_files]
    })
    
    # Merge with training data
    train_df_original_count = len(train_df)
    train_df = pd.concat([train_df, whatsapp_df], ignore_index=True)
    
    print(f"  Added {len(whatsapp_df)} WhatsApp voice notes to training set")
    print(f"  ASVspoof samples: {train_df_original_count}")
    print(f"  WhatsApp samples: {len(whatsapp_df)}")
    print(f"  Total:            {len(train_df)}")
else:
    print("No WhatsApp voice notes added (directory not found)")

print(f"\nFinal dataset:")
print(f"Training samples:   {len(train_df)}")
print(f"Dev samples:        {len(dev_df)}")
print(f"Train label split:  {train_df['label'].value_counts().to_dict()}")
print(f"Dev label split:    {dev_df['label'].value_counts().to_dict()}")

  Added 53 WhatsApp voice notes to training set
  ASVspoof samples: 25380
  WhatsApp samples: 53
  Total:            25433

Final dataset:
Training samples:   25433
Dev samples:        24844
Train label split:  {1: 22800, 0: 2633}
Dev label split:    {1: 22296, 0: 2548}


In [ ]:
# Cell 5: Dataset with WhatsApp augmentation and logging

import os
import tempfile
import subprocess
import hashlib
import numpy as np
import torch
import random
import soundfile as sf
import time
from torch.utils.data import Dataset
from scipy import signal
from collections import defaultdict

SAMPLE_RATE = 16000
MAX_SAMPLES = 64000  # 4 seconds


# Debug Settings
DEBUG = True
OPUS_DEBUG = False      
FORCE_OPUS = False
DISABLE_CACHE = True

OPUS_CACHE_DIR = None if DISABLE_CACHE else os.environ.get("OPUS_CACHE_DIR", None)

if OPUS_CACHE_DIR:
    os.makedirs(OPUS_CACHE_DIR, exist_ok=True)
    print(f"Opus compression caching enabled at: {OPUS_CACHE_DIR}")
else:
    print("Opus cache disabled")

print("\n=== AUGMENTATION STRATEGY ===")
print("1. Opus codec compression (85%)")
print("2. Bandpass filter 50-400Hz")
print("3. Background noise")
print("4. Volume variation")
print("5. Pitch shifting (±3 semitones)")
print("============================\n")

# Global Stats Tracker
AUG_STATS = defaultdict(int)
AUG_TOTAL = 0


class ASVspoofDataset(Dataset):

    def __init__(self, df, feature_extractor, augment=True):
        self.df = df
        self.feature_extractor = feature_extractor
        self.augment = augment

    def __len__(self):
        return len(self.df)

    # Auto Loader
    def load_audio(self, path):

        if path.endswith(".ogg"):
            tmp_wav = tempfile.mktemp(suffix=".wav")

            try:
                result = subprocess.run([
                    "ffmpeg", "-y", "-loglevel", "error",
                    "-i", path,
                    "-ar", str(SAMPLE_RATE),
                    "-ac", "1",
                    tmp_wav
                ], capture_output=True, text=True)

                if result.returncode != 0:
                    raise RuntimeError(result.stderr)

                audio, _ = sf.read(tmp_wav, dtype="float32")
                return audio

            finally:
                if os.path.exists(tmp_wav):
                    os.remove(tmp_wav)

        else:
            audio, _ = sf.read(path, dtype="float32")
            return audio

    # Bandpass Filter
    def apply_bandpass_filter(self, audio, sr=16000, lowcut=50, highcut=400):
        nyquist = sr / 2
        low = lowcut / nyquist
        high = highcut / nyquist

        sos = signal.butter(5, [low, high], btype='band', output='sos')
        return signal.sosfilt(sos, audio).astype(np.float32)

    # Opus Compression
    def apply_opus_compression(self, audio_tensor, sr=16000, source_path=None):

        if OPUS_CACHE_DIR and source_path:
            cache_key = hashlib.md5(source_path.encode()).hexdigest()
            cache_file = os.path.join(OPUS_CACHE_DIR, f"{cache_key}.wav")

            if os.path.exists(cache_file):
                audio, _ = sf.read(cache_file, dtype="float32")
                return torch.from_numpy(audio).unsqueeze(0)

        audio_np = audio_tensor.squeeze(0).numpy()

        tmp_in = tempfile.mktemp(suffix=".wav")
        tmp_ogg = tempfile.mktemp(suffix=".ogg")
        tmp_out = tempfile.mktemp(suffix=".wav")

        try:
            sf.write(tmp_in, audio_np, sr)

            bitrate = random.choice([12, 16, 24])

            start_time = time.time()

            # Encode
            result = subprocess.run([
                "ffmpeg", "-y", "-loglevel", "error",
                "-i", tmp_in,
                "-c:a", "libopus",
                "-b:a", f"{bitrate}k",
                "-ar", str(sr),
                tmp_ogg
            ], capture_output=True, text=True)

            if result.returncode != 0:
                raise RuntimeError(result.stderr)

            # Decode
            result = subprocess.run([
                "ffmpeg", "-y", "-loglevel", "error",
                "-i", tmp_ogg,
                "-ar", str(sr),
                "-ac", "1",
                tmp_out
            ], capture_output=True, text=True)

            if result.returncode != 0:
                raise RuntimeError(result.stderr)

            compressed_audio, _ = sf.read(tmp_out, dtype="float32")

            elapsed = time.time() - start_time

            # Logging
            if DEBUG and OPUS_DEBUG:
                print(f"[OPUS] {os.path.basename(source_path)} | {bitrate}kbps | {elapsed:.3f}s")

            if OPUS_CACHE_DIR and source_path:
                sf.write(cache_file, compressed_audio, sr)

            return torch.from_numpy(compressed_audio).unsqueeze(0)

        finally:
            for f in [tmp_in, tmp_ogg, tmp_out]:
                if f and os.path.exists(f):
                    os.remove(f)

    # Pitch Shift
    def apply_pitch_shift(self, audio, semitones=0):
        if semitones == 0:
            return audio

        rate = 2 ** (semitones / 12)
        indices = np.round(np.arange(0, len(audio), rate)).astype(int)
        indices = indices[indices < len(audio)]
        return audio[indices]

    # Main Pipeline
    def __getitem__(self, idx):

        global AUG_STATS, AUG_TOTAL
        AUG_TOTAL += 1

        row = self.df.iloc[idx]

        audio = self.load_audio(row["path"]).astype(np.float32)
        audio_tensor = torch.from_numpy(audio).unsqueeze(0)

        is_whatsapp = row["path"].endswith(".ogg")

        if self.augment and not is_whatsapp:

            if FORCE_OPUS or random.random() < 0.85:
                audio_tensor = self.apply_opus_compression(
                    audio_tensor, SAMPLE_RATE, row["path"]
                )
                AUG_STATS["opus"] += 1

            audio = audio_tensor.squeeze(0).numpy()

            if random.random() < 0.60:
                highcut = random.randint(350, 450)
                audio = self.apply_bandpass_filter(audio, SAMPLE_RATE, 50, highcut)
                AUG_STATS["bandpass"] += 1

            if random.random() < 0.50:
                noise = np.random.randn(len(audio)) * random.uniform(0.003, 0.02)
                audio += noise.astype(np.float32)
                AUG_STATS["noise"] += 1

            if random.random() < 0.40:
                audio *= random.uniform(0.5, 1.5)
                AUG_STATS["volume"] += 1

            if random.random() < 0.30:
                semitones = random.uniform(-3, 3)
                audio = self.apply_pitch_shift(audio, semitones)
                AUG_STATS["pitch"] += 1

            audio_tensor = torch.from_numpy(audio).unsqueeze(0)

        audio = audio_tensor.squeeze(0).numpy()

        if len(audio) >= MAX_SAMPLES:
            audio = audio[:MAX_SAMPLES]
        else:
            audio = np.pad(audio, (0, MAX_SAMPLES - len(audio)))

        peak = np.abs(audio).max()
        if peak > 0:
            audio = audio / peak

        inputs = self.feature_extractor(
            audio,
            sampling_rate=SAMPLE_RATE,
            return_tensors="pt",
            padding=False
        )

        return {
            "input_values": inputs["input_values"].squeeze(0).to(torch.bfloat16),
            "label": torch.tensor(row["label"], dtype=torch.long)
        }

⚠️ Opus cache disabled (DEBUG mode)

=== AUGMENTATION STRATEGY ===
1. Opus codec compression (85%)
2. Bandpass filter 50-400Hz
3. Background noise
4. Volume variation
5. Pitch shifting (±3 semitones)



In [ ]:
# Cell 6: Load model with proper label mapping
MODEL_NAME = "facebook/wav2vec2-base"
device     = torch.device("cuda")
 
print("Loading feature extractor...")
feature_extractor = Wav2Vec2FeatureExtractor.from_pretrained(MODEL_NAME)
 
print("Loading model...")
model = Wav2Vec2ForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    ignore_mismatched_sizes=True
)
 
# Set proper label names
model.config.id2label = {0: "bonafide", 1: "spoof"}
model.config.label2id = {"bonafide": 0, "spoof": 1}
 
# Freeze the CNN feature extractor
for param in model.wav2vec2.feature_extractor.parameters():
    param.requires_grad = False
 
# Freeze the bottom 6 transformer layers (half of 12)
for i in range(6):
    for param in model.wav2vec2.encoder.layers[i].parameters():
        param.requires_grad = False
 
# Move model to GPU then convert weights to bfloat16
model = model.to(device)
model = model.to(torch.bfloat16)
 
# Allow TF32
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
 
print(f"Model loaded on {device}")
 
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"Trainable parameters: {trainable:,} / {total:,}")
print(f"Frozen parameters:    {total - trainable:,} / {total:,}")
print(f"\nLabel mapping: {model.config.id2label}")

Loading feature extractor...
Loading model...


Loading weights:   0%|          | 0/211 [00:00<?, ?it/s]

Wav2Vec2ForSequenceClassification LOAD REPORT from: facebook/wav2vec2-base
Key                          | Status     | 
-----------------------------+------------+-
project_q.bias               | UNEXPECTED | 
project_q.weight             | UNEXPECTED | 
quantizer.weight_proj.weight | UNEXPECTED | 
project_hid.weight           | UNEXPECTED | 
project_hid.bias             | UNEXPECTED | 
quantizer.weight_proj.bias   | UNEXPECTED | 
quantizer.codevectors        | UNEXPECTED | 
classifier.bias              | MISSING    | 
classifier.weight            | MISSING    | 
projector.weight             | MISSING    | 
projector.bias               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model loaded on cuda
Trainable parameters: 47,841,410 / 94,569,090
Frozen parameters:    46,727,680 / 94,569,090

Label mapping: {0: 'bonafide', 1: 'spoof'}


In [ ]:
# Cell 7: Create data loaders
from torch.utils.data import WeightedRandomSampler
 
# Create dataset objects with augmentation
train_dataset = ASVspoofDataset(train_df, feature_extractor, augment=True)
dev_dataset   = ASVspoofDataset(dev_df,   feature_extractor, augment=False)  # No augmentation for validation
 
# Handle class imbalance using a weighted sampler
class_counts  = train_df["label"].value_counts().sort_index().values
class_weights = 1.0 / class_counts
sample_weights = train_df["label"].map({0: class_weights[0], 1: class_weights[1]}).values

sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(train_dataset),
    replacement=True  # Changed to True for better class balance
)
 
# DataLoaders
train_loader = DataLoader(
    train_dataset,
    batch_size=8,
    sampler=sampler,
    num_workers=0,
    pin_memory=True
)
 
dev_loader = DataLoader(
    dev_dataset,
    batch_size=8,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)
 
print(f"Training batches:   {len(train_loader)}")
print(f"Development batches: {len(dev_loader)}")
print("Improved augmentation enabled for training set")
print(f"Class weights: bonafide={class_weights[0]:.4f}, spoof={class_weights[1]:.4f}")

Training batches:   3180
Development batches: 3106
✓ IMPROVED augmentation enabled for training set
✓ Class weights: bonafide=0.0004, spoof=0.0000


In [46]:
# Cell 8: Optimiser and evaluation function
from torch.optim import AdamW
from torch.optim.lr_scheduler import LinearLR
from sklearn.metrics import roc_auc_score
 
# AdamW optimiser
optimiser = AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=1e-4,
    weight_decay=0.01
)
 
# Linear warmup scheduler
total_steps  = len(train_loader) * 5  # 5 epochs
warmup_steps = int(0.1 * total_steps)
scheduler = LinearLR(
    optimiser,
    start_factor=0.1,
    end_factor=1.0,
    total_iters=warmup_steps
)
 
def evaluate(model, loader, device):
    """Run model on dev set and return loss, accuracy and ROC-AUC."""
    model.eval()
    total_loss, correct, all_labels, all_probs = 0, 0, [], []
 
    with torch.no_grad():
        for batch in loader:
            input_values = batch["input_values"].to(device)
            labels       = batch["label"].to(device)
 
            outputs = model(input_values=input_values, labels=labels)
            total_loss += outputs.loss.item()
 
            probs  = torch.softmax(outputs.logits, dim=-1).to(torch.float32)
            preds  = probs.argmax(dim=-1)
            correct += (preds == labels).sum().item()
 
            all_labels.extend(labels.cpu().numpy())
            all_probs.extend(probs[:, 1].cpu().numpy())
 
    avg_loss = total_loss / len(loader)
    accuracy = correct / len(loader.dataset)
    roc_auc  = roc_auc_score(all_labels, all_probs)
    return avg_loss, accuracy, roc_auc
 
print("Optimiser and evaluation function ready")
print(f"Total training steps: {total_steps}")
print(f"Warmup steps:         {warmup_steps}")

Optimiser and evaluation function ready
Total training steps: 15900
Warmup steps:         1590


In [ ]:
# Cell 9: Training loop with augmentation statistics reporting

from torch.amp import autocast
from sklearn.metrics import roc_auc_score

EPOCHS       = 5
EVAL_STEPS   = 200
SAVE_DIR     = r"C:\deepfake-project\models\wav2vec2_finetuned_v2"
best_roc_auc = 0.0
patience     = 0
PATIENCE_MAX = 4

os.makedirs(SAVE_DIR, exist_ok=True)

print("\n" + "="*60)
print("TRAINING")
print("="*60)
print(f"Save directory: {SAVE_DIR}")
print(f"Eval frequency: every {EVAL_STEPS} steps")
print(f"Early stopping: {PATIENCE_MAX} evaluations without improvement")
print("="*60 + "\n")


for epoch in range(EPOCHS):

    model.train()
    epoch_loss = 0
    step = 0

    # reset augmentation stats each epoch
    AUG_STATS.clear()
    AUG_TOTAL = 0

    print(f"\n================ EPOCH {epoch+1} START ================\n")

    for batch in train_loader:

        input_values = batch["input_values"].to(device)
        labels       = batch["label"].to(device)

        with autocast(device_type="cuda", dtype=torch.bfloat16):
            outputs = model(input_values=input_values, labels=labels)

        loss = outputs.loss

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimiser.step()
        scheduler.step()
        optimiser.zero_grad()

        epoch_loss += loss.item()
        step += 1

        if step % 50 == 0:
            avg_loss = epoch_loss / step
            print(f"Epoch {epoch+1} | Step {step}/{len(train_loader)} | Loss: {avg_loss:.4f}")

        # Evaluation
        if step % EVAL_STEPS == 0:

            dev_loss, dev_acc, dev_roc = evaluate(model, dev_loader, device)

            print(f"\n>>> Eval @ step {step} | Loss: {dev_loss:.4f} | Acc: {dev_acc:.4f} | ROC-AUC: {dev_roc:.4f}")

            if dev_roc > best_roc_auc:
                best_roc_auc = dev_roc
                patience = 0
                model.save_pretrained(SAVE_DIR)
                feature_extractor.save_pretrained(SAVE_DIR)
                print(f"     New best model saved (ROC-AUC: {best_roc_auc:.4f})")
            else:
                patience += 1
                print(f"    No improvement — patience {patience}/{PATIENCE_MAX}")
                if patience >= PATIENCE_MAX:
                    print("\nEarly stopping triggered")
                    break

            model.train()

    # Augmentation Stats Report
    print("\n================ AUGMENTATION STATS ================")
    print(f"Total augmented samples: {AUG_TOTAL}")

    if AUG_TOTAL > 0:
        for k, v in AUG_STATS.items():
            pct = (v / AUG_TOTAL) * 100
            print(f"{k}: {v} ({pct:.1f}%)")
    else:
        print("No augmentation stats recorded")

    print("====================================================\n")

    print(f"\nEpoch {epoch+1} complete | Avg loss: {epoch_loss/len(train_loader):.4f}\n")

    if patience >= PATIENCE_MAX:
        break


print("\n" + "="*60)
print(f"TRAINING COMPLETE | Best ROC-AUC: {best_roc_auc:.4f}")
print(f"Best model saved to: {SAVE_DIR}")
print("="*60)


TRAINING WITH IMPROVED WHATSAPP-STYLE AUGMENTATION
Save directory: C:\deepfake-project\models\wav2vec2_finetuned_v2
Eval frequency: every 200 steps
Early stopping: 4 evaluations without improvement


================ EPOCH 1 START ================

Epoch 1 | Step 50/3180 | Loss: 0.6942
Epoch 1 | Step 100/3180 | Loss: 0.6933
Epoch 1 | Step 150/3180 | Loss: 0.6934
Epoch 1 | Step 200/3180 | Loss: 0.6926

>>> Eval @ step 200 | Loss: 0.6673 | Acc: 0.9014 | ROC-AUC: 0.7766


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

    ✓ New best model saved (ROC-AUC: 0.7766)
Epoch 1 | Step 250/3180 | Loss: 0.6915
Epoch 1 | Step 300/3180 | Loss: 0.6906
Epoch 1 | Step 350/3180 | Loss: 0.6892
Epoch 1 | Step 400/3180 | Loss: 0.6859

>>> Eval @ step 400 | Loss: 0.6516 | Acc: 0.7120 | ROC-AUC: 0.9100


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

    ✓ New best model saved (ROC-AUC: 0.9100)
Epoch 1 | Step 450/3180 | Loss: 0.6754
Epoch 1 | Step 500/3180 | Loss: 0.6582
Epoch 1 | Step 550/3180 | Loss: 0.6415
Epoch 1 | Step 600/3180 | Loss: 0.6209

>>> Eval @ step 600 | Loss: 0.4222 | Acc: 0.8348 | ROC-AUC: 0.9524


...

    ✓ New best model saved (ROC-AUC: 0.9985)
Epoch 2 | Step 850/3180 | Loss: 0.0993
Epoch 2 | Step 900/3180 | Loss: 0.0971
Epoch 2 | Step 950/3180 | Loss: 0.0965
Epoch 2 | Step 1000/3180 | Loss: 0.0950

>>> Eval @ step 1000 | Loss: 0.0567 | Acc: 0.9845 | ROC-AUC: 0.9987


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

    ✓ New best model saved (ROC-AUC: 0.9987)
Epoch 2 | Step 1050/3180 | Loss: 0.0931
Epoch 2 | Step 1100/3180 | Loss: 0.0898
Epoch 2 | Step 1150/3180 | Loss: 0.0893
Epoch 2 | Step 1200/3180 | Loss: 0.0904

>>> Eval @ step 1200 | Loss: 0.1020 | Acc: 0.9738 | ROC-AUC: 0.9983
    No improvement — patience 1/4
Epoch 2 | Step 1250/3180 | Loss: 0.0888
Epoch 2 | Step 1300/3180 | Loss: 0.0900
Epoch 2 | Step 1350/3180 | Loss: 0.0880
Epoch 2 | Step 1400/3180 | Loss: 0.0874

>>> Eval @ step 1400 | Loss: 0.0947 | Acc: 0.9760 | ROC-AUC: 0.9985
    No improvement — patience 2/4
Epoch 2 | Step 1450/3180 | Loss: 0.0874
Epoch 2 | Step 1500/3180 | Loss: 0.0870
Epoch 2 | Step 1550/3180 | Loss: 0.0870
Epoch 2 | Step 1600/3180 | Loss: 0.0865

>>> Eval @ step 1600 | Loss: 0.0406 | Acc: 0.9891 | ROC-AUC: 0.9991


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

    ✓ New best model saved (ROC-AUC: 0.9991)
Epoch 2 | Step 1650/3180 | Loss: 0.0854
Epoch 2 | Step 1700/3180 | Loss: 0.0854
Epoch 2 | Step 1750/3180 | Loss: 0.0852
Epoch 2 | Step 1800/3180 | Loss: 0.0837

>>> Eval @ step 1800 | Loss: 0.0225 | Acc: 0.9944 | ROC-AUC: 0.9990
    No improvement — patience 1/4
Epoch 2 | Step 1850/3180 | Loss: 0.0832
Epoch 2 | Step 1900/3180 | Loss: 0.0833
Epoch 2 | Step 1950/3180 | Loss: 0.0825
Epoch 2 | Step 2000/3180 | Loss: 0.0827

>>> Eval @ step 2000 | Loss: 0.1086 | Acc: 0.9724 | ROC-AUC: 0.9990
    No improvement — patience 2/4
Epoch 2 | Step 2050/3180 | Loss: 0.0825
Epoch 2 | Step 2100/3180 | Loss: 0.0815
Epoch 2 | Step 2150/3180 | Loss: 0.0798
Epoch 2 | Step 2200/3180 | Loss: 0.0799

>>> Eval @ step 2200 | Loss: 0.0788 | Acc: 0.9801 | ROC-AUC: 0.9988
    No improvement — patience 3/4
Epoch 2 | Step 2250/3180 | Loss: 0.0791
Epoch 2 | Step 2300/3180 | Loss: 0.0793
Epoch 2 | Step 2350/3180 | Loss: 0.0784
Epoch 2 | Step 2400/3180 | Loss: 0.0789

>>> E

In [48]:
# Cell 10: Test saved model
saved_model = Wav2Vec2ForSequenceClassification.from_pretrained(SAVE_DIR)
saved_extractor = Wav2Vec2FeatureExtractor.from_pretrained(SAVE_DIR)
 
saved_model = saved_model.to(device)
saved_model = saved_model.to(torch.bfloat16)
saved_model.eval()
 
print(f"\n=== Saved Model Configuration ===")
print(f"id2label: {saved_model.config.id2label}")
print(f"label2id: {saved_model.config.label2id}")
 
def predict(audio_path):
    audio, sr = sf.read(audio_path)
    audio = audio.astype(np.float32)
 
    if len(audio) >= MAX_SAMPLES:
        audio = audio[:MAX_SAMPLES]
    else:
        audio = np.pad(audio, (0, MAX_SAMPLES - len(audio)))
 
    peak = np.abs(audio).max()
    if peak > 0:
        audio = audio / peak
 
    inputs = saved_extractor(
        audio,
        sampling_rate=SAMPLE_RATE,
        return_tensors="pt",
        padding=False
    )
 
    input_values = inputs["input_values"].to(device).to(torch.bfloat16)
 
    with torch.no_grad():
        outputs = saved_model(input_values=input_values)
        probs = torch.softmax(outputs.logits, dim=-1).to(torch.float32)
 
    return {
        "bonafide": round(probs[0][0].item(), 4),
        "spoof":    round(probs[0][1].item(), 4),
        "verdict":  "REAL" if probs[0][0] > probs[0][1] else "FAKE"
    }
 
# Test on dev set
real_file = dev_df[dev_df["label"] == 0].iloc[0]["path"]
fake_file = dev_df[dev_df["label"] == 1].iloc[0]["path"]
 
print("\n=== Testing saved model ===")
print(f"\nReal audio: {os.path.basename(real_file)}")
print(predict(real_file))
 
print(f"\nFake audio: {os.path.basename(fake_file)}")
print(predict(fake_file))

Loading weights:   0%|          | 0/215 [00:00<?, ?it/s]


=== Saved Model Configuration ===
id2label: {0: 'bonafide', 1: 'spoof'}
label2id: {'bonafide': 0, 'spoof': 1}

=== Testing saved model ===

Real audio: LA_D_1047731.flac
{'bonafide': 0.9961, 'spoof': 0.0021, 'verdict': 'REAL'}

Fake audio: LA_D_1008730.flac
{'bonafide': 0.0015, 'spoof': 1.0, 'verdict': 'FAKE'}
